In [ ]:
# Set up dependencies

!pip3 install torch transformers datasets scikit-learn numpy gdown --break-system-packages

In [ ]:
# Download the pre-trained model of choice
import gdown

# https://drive.google.com/file/d/14f1qxGU9Hxaa4BcY5mHwGtWTWEe6ncU3/view?usp=sharing # V3 best
GOOGLE_DRIVE_FILE_ID = '14f1qxGU9Hxaa4BcY5mHwGtWTWEe6ncU3'
OUTPUT_FILE_NAME = 'model.pth'

gdown.download(id=GOOGLE_DRIVE_FILE_ID, output=OUTPUT_FILE_NAME, quiet=False)

In [ ]:
# Load model and tokenizer

import torch
import torch.nn as nn
import math
from transformers import GPT2Tokenizer


class SelfAttention(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads

        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        N, seq_len, _ = x.shape

        values = self.values(x)
        keys = self.keys(x)
        queries = self.queries(x)

        values = values.reshape(N, seq_len, self.heads, self.head_dim).transpose(1, 2)
        keys = keys.reshape(N, seq_len, self.heads, self.head_dim).transpose(1, 2)
        queries = queries.reshape(N, seq_len, self.heads, self.head_dim).transpose(1, 2)

        energy = torch.matmul(queries, keys.transpose(-2, -1))
        energy = energy / math.sqrt(self.head_dim)

        mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)
        energy = energy.masked_fill(mask == 0, float("-inf"))

        attention = torch.softmax(energy, dim=-1)
        out = torch.matmul(attention, values)

        out = out.transpose(1, 2).contiguous().reshape(N, seq_len, self.embed_size)
        return self.fc_out(out)


class TransformerBlock(nn.Module):
    def __init__(self, embed_size, heads):
        super().__init__()
        self.attention = SelfAttention(embed_size, heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.norm2 = nn.LayerNorm(embed_size)

        self.ff = nn.Sequential(
            nn.Linear(embed_size, 4 * embed_size),
            nn.GELU(),
            nn.Linear(4 * embed_size, embed_size)
        )

    def forward(self, x):
        x = self.norm1(x + self.attention(x))
        x = self.norm2(x + self.ff(x))
        return x

class GPT1(nn.Module):
    def __init__(self, vocab_size, embed_size, num_layers, heads, max_len):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_len, embed_size)

        self.layers = nn.ModuleList(
            [TransformerBlock(embed_size, heads) for _ in range(num_layers)]
        )

        self.norm = nn.LayerNorm(embed_size)
        self.fc_out = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        N, seq_len = x.shape
        positions = torch.arange(0, seq_len).expand(N, seq_len).to(x.device)

        x = self.token_embedding(x) + self.position_embedding(positions)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        return self.fc_out(x)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = GPT1(
    vocab_size = tokenizer.vocab_size,
    embed_size = 256,
    num_layers = 4,
    heads = 4,
    max_len = 512
).to(device)

In [ ]:
# Load the state model from the downloaded checkpoint

state_dict = torch.load(OUTPUT_FILE_NAME, map_location=device)
model.load_state_dict(state_dict['model_state_dict'])
model.eval()

In [ ]:
# Load the NRI dataset
from datasets import load_dataset

# dataset = load_dataset("snli")
dataset = load_dataset("stanfordnlp/snli")
test_set = dataset["test"].filter(lambda x: x["label"] != -1)  # drop unlabeled
subset = test_set.select(range(500))  # quick pilot run first

print(f"Dataset length === {len(test_set)}")
print(f"test_set length === {len(test_set)}")
print(f"subset length === {len(subset)}")
print(test_set)


In [ ]:
unique_labels = sorted(list(set(subset['label'])))
print(f"Unique labels in the subset: {unique_labels}")

# 0 : Entailment
# 1 : Neutral
# 2 : Contradiction

In [ ]:
# Baseline

distribution = subset

from collections import Counter
label_counts = Counter(distribution["label"])
majority_label = label_counts.most_common(1)[0][0]
majority_baseline = label_counts[majority_label] / len(distribution)
print(f"Majority-class baseline: {majority_baseline:.2%}")

num_unique_labels = len(unique_labels)
random_chance_baseline = 1 / num_unique_labels
print(f"Random-chance baseline: {random_chance_baseline:.2%}")

In [ ]:
# Run perplexity-based scoring

import torch.nn.functional as F

label_words = {0: " True", 1: " Neither", 2: " False"}  # entailment/neutral/contradiction

def score_continuation(prompt, continuation):
    full = prompt + continuation
    input_ids = tokenizer(full, return_tensors="pt").input_ids.to(device)
    prompt_len = tokenizer(prompt, return_tensors="pt").input_ids.shape[1]

    with torch.no_grad():
        logits = model(input_ids)

    log_probs = F.log_softmax(logits[0, prompt_len-1:-1], dim=-1)
    target_ids = input_ids[0, prompt_len:]
    return log_probs[range(len(target_ids)), target_ids].sum().item()

correct = 0
for ex in distribution:
    prompt = f"{ex['premise']} Question: {ex['hypothesis']} True, False, or Neither? Answer:"
    scores = {label: score_continuation(prompt, word) for label, word in label_words.items()}
    pred = max(scores, key=scores.get)
    correct += int(pred == ex["label"])

perplexity_accuracy = correct/len(distribution)
print(f"Perplexity-scoring accuracy: {perplexity_accuracy:.2%}")

In [ ]:
# Perplexity Few shot scoring

import torch.nn.functional as F
import random

label_words = {0: " True", 1: " Neither", 2: " False"}  # entailment/neutral/contradiction

def format_example(premise, hypothesis, answer=None):
    prompt = f"{premise} Question: {hypothesis} True, False, or Neither? Answer:"
    if answer is not None:
        prompt += answer  # answer already has leading space, e.g. " True"
    return prompt

def build_few_shot_prefix(train_data, n_per_class=1, seed=42):
    """Pick n_per_class examples for each label from train_data, format them, and join."""
    random.seed(seed)
    by_label = {0: [], 1: [], 2: []}
    for ex in train_data:
        if ex["label"] in by_label:
            by_label[ex["label"]].append(ex)

    shots = []
    for label, examples in by_label.items():
        chosen = random.sample(examples, n_per_class)
        for ex in chosen:
            shots.append(format_example(ex["premise"], ex["hypothesis"], label_words[label]))

    random.shuffle(shots)  # avoid the model just learning label order
    return "\n\n".join(shots) + "\n\n"

def score_continuation(prompt, continuation):
    full = prompt + continuation
    input_ids = tokenizer(full, return_tensors="pt").input_ids.to(device)
    prompt_len = tokenizer(prompt, return_tensors="pt").input_ids.shape[1]

    with torch.no_grad():
        logits = model(input_ids)

    log_probs = F.log_softmax(logits[0, prompt_len-1:-1], dim=-1)
    target_ids = input_ids[0, prompt_len:]
    return log_probs[range(len(target_ids)), target_ids].sum().item()

# Build the few-shot prefix once, from the training split (NOT from your test/eval set)
few_shot_prefix = build_few_shot_prefix(dataset["train"], n_per_class=1, seed=11)
print("Few-shot prefix:\n", few_shot_prefix)

correct = 0
for ex in distribution:
    base_prompt = f"{ex['premise']} Question: {ex['hypothesis']} True, False, or Neither? Answer:"
    prompt = few_shot_prefix + base_prompt
    scores = {label: score_continuation(prompt, word) for label, word in label_words.items()}
    pred = max(scores, key=scores.get)
    correct += int(pred == ex["label"])

few_shot_accuracy = correct/len(distribution)
print(f"Few-shot perplexity-scoring accuracy: {few_shot_accuracy:.2%}")

In [ ]:
# Layer probe

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

def get_embedding_at_layer(text, layer_idx):
    ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).input_ids.to(device)
    N, seq_len = ids.shape
    positions = torch.arange(0, seq_len).expand(N, seq_len).to(device)

    with torch.no_grad():
        x = model.token_embedding(ids) + model.position_embedding(positions)

        for i, layer in enumerate(model.layers):
            x = layer(x)
            if i == layer_idx:
                break # Get the output after the specified layer

        # Apply the final layer normalization if we are getting the last layer's embedding
        # Otherwise, the output after the TransformerBlock is the hidden state
        hidden_state = model.norm(x) if layer_idx == len(model.layers) - 1 else x

    return hidden_state.mean(dim=1).squeeze().cpu().numpy()

# Loop over layer_idx = 0..num_layers-1 and compare probing accuracy per layer
print("Probing accuracy per layer:")
global_layer_accuracies = []
global_layer_indices = []

for layer_idx in range(len(model.layers)):
    X_layer, y_layer = [], []
    for ex in distribution:
        p_emb_layer = get_embedding_at_layer(ex["premise"], layer_idx)
        h_emb_layer = get_embedding_at_layer(ex["hypothesis"], layer_idx)
        feat_layer = np.concatenate([p_emb_layer, h_emb_layer, np.abs(p_emb_layer - h_emb_layer), p_emb_layer * h_emb_layer])
        X_layer.append(feat_layer)
        y_layer.append(ex["label"])

    X_train_layer, X_val_layer, y_train_layer, y_val_layer = train_test_split(X_layer, y_layer, test_size=0.2, random_state=42)
    clf_layer = LogisticRegression(max_iter=5000, multi_class="multinomial").fit(X_train_layer, y_train_layer)
    accuracy = clf_layer.score(X_val_layer, y_val_layer)
    global_layer_accuracies.append(accuracy)
    global_layer_indices.append(layer_idx)
    print(f"Layer {layer_idx} accuracy: {accuracy:.2%}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use the globally available lists populated in the previous step
layer_accuracies = global_layer_accuracies
layer_indices = global_layer_indices

# Create the plot
plt.figure(figsize=(10, 6))
sns.barplot(x=layer_indices, y=layer_accuracies, palette='viridis')
plt.title('Probing Accuracy Across Transformer Layers')
plt.xlabel('Layer Index')
plt.ylabel('Probing Accuracy')
plt.ylim(min(layer_accuracies) * 0.9, max(layer_accuracies) * 1.1) # Adjust y-axis limits for better visualization
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Construct the data for the summary table
summary_data = []

summary_data.append({
    "Method": "Perplexity scoring",
    "Accuracy": perplexity_accuracy,
    "Baseline (majority)": majority_baseline,
    "Baseline (random)": random_chance_baseline
})

for idx, acc in zip(global_layer_indices, global_layer_accuracies):
    summary_data.append({
        "Method": f"Probing (Layer {idx})",
        "Accuracy": acc,
        "Baseline (majority)": majority_baseline,
        "Baseline (random)": random_chance_baseline
    })

# Create and display the DataFrame
summary_df = pd.DataFrame(summary_data)

print("Summary of All Evaluation Results:")
display(summary_df.style.format({
    'Accuracy': '{:.2%}',
    'Baseline (majority)': '{:.2%}',
    'Baseline (random)': '{:.2%}'
}))